# Import raw Pulse Eco measurements

This notebook stores raw Pulse Eco PM10/PM2.5 measurements from `feature_engineering/pulse_data/` into SQLite.

It does **not** resample, interpolate, forward-fill, back-fill, or median-fill values. Missing hours remain missing because no row is created for them.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

In [2]:
BASE_DIR = Path.cwd()
if BASE_DIR.name == "feature_engineering":
    BASE_DIR = BASE_DIR.parent

PULSE_DATA_DIR = BASE_DIR / "feature_engineering" / "pulse_data"
DB_PATH = BASE_DIR / "data" / "bitola.db"
CITY = "Bitola"

# Keep this focused on observed pollution values for dashboard comparison.
# Set MEASUREMENT_TYPES = None if you later want to store every Pulse Eco type.
MEASUREMENT_TYPES = ["pm10", "pm25"]

print("Pulse data:", PULSE_DATA_DIR)
print("Database:", DB_PATH)

Pulse data: /mnt/c/Users/RazorVision/Desktop/project-vrnmp/feature_engineering/pulse_data
Database: /mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/bitola.db


In [3]:
def load_pulse_measurements(pulse_data_dir, measurement_types=None):
    pulse_files = sorted(pulse_data_dir.rglob("*.csv"))
    if not pulse_files:
        raise FileNotFoundError(f"No Pulse Eco CSV files found under {pulse_data_dir}")

    frames = []
    for path in pulse_files:
        df = pd.read_csv(path, usecols=["timestamp", "sensorId", "lat", "lon", "type", "value"])

        if measurement_types is not None:
            df = df[df["type"].isin(measurement_types)].copy()

        if df.empty:
            continue

        df["city"] = CITY
        df["sensor_id"] = df["sensorId"].astype(str)
        df["measurement_type"] = df["type"].astype(str)
        df["timestamp_local"] = df["timestamp"].astype(str)
        df["timestamp_utc"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
        df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
        df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["source_file"] = path.relative_to(BASE_DIR).as_posix()

        df = df.dropna(subset=["timestamp_utc", "sensor_id", "measurement_type", "value"])
        frames.append(
            df[[
                "city",
                "sensor_id",
                "timestamp_utc",
                "timestamp_local",
                "measurement_type",
                "value",
                "lat",
                "lon",
                "source_file",
            ]]
        )

    if not frames:
        raise ValueError("No matching Pulse Eco measurement rows found")

    measurements = pd.concat(frames, ignore_index=True)
    measurements["timestamp_utc"] = measurements["timestamp_utc"].dt.strftime("%Y-%m-%d %H:%M:%S")

    # Weekly downloads can overlap at boundaries. Keep one raw row per sensor/time/type.
    measurements = measurements.drop_duplicates(
        subset=["city", "sensor_id", "timestamp_utc", "measurement_type"],
        keep="last",
    )

    return measurements.sort_values(["sensor_id", "timestamp_utc", "measurement_type"]).reset_index(drop=True)

In [4]:
raw_measurements = load_pulse_measurements(PULSE_DATA_DIR, MEASUREMENT_TYPES)

print("Rows:", len(raw_measurements))
print("Sensors:", raw_measurements["sensor_id"].nunique())
print("Timestamp range:", raw_measurements["timestamp_utc"].min(), "to", raw_measurements["timestamp_utc"].max())

raw_measurements.head()

Rows: 154721
Sensors: 12
Timestamp range: 2025-12-01 00:00:43 to 2026-03-01 23:59:38


,city,sensor_id,timestamp_utc,timestamp_local,measurement_type,value,lat,lon,source_file
0,Bitola,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 00:04:26,2025-12-01T01:04:26+01:00,pm10,20,41.023537,21.330127,feature_engineering/pulse_data/2025/week_1/168...
1,Bitola,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 00:04:26,2025-12-01T01:04:26+01:00,pm25,12,41.023537,21.330127,feature_engineering/pulse_data/2025/week_1/168...
2,Bitola,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 00:19:57,2025-12-01T01:19:57+01:00,pm10,68,41.023537,21.330127,feature_engineering/pulse_data/2025/week_1/168...
3,Bitola,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 00:19:57,2025-12-01T01:19:57+01:00,pm25,31,41.023537,21.330127,feature_engineering/pulse_data/2025/week_1/168...
4,Bitola,16836a55-7140-43e2-9a63-56fac5cba714,2025-12-01 00:35:28,2025-12-01T01:35:28+01:00,pm10,27,41.023537,21.330127,feature_engineering/pulse_data/2025/week_1/168...


In [5]:
raw_measurements.groupby("measurement_type").agg(
    rows=("value", "size"),
    sensors=("sensor_id", "nunique"),
    min_value=("value", "min"),
    median_value=("value", "median"),
    max_value=("value", "max"),
).round(2)

,rows,sensors,min_value,median_value,max_value
measurement_type,,,,,
pm10,77280,12,0,17.0,1926
pm25,77441,12,0,9.0,994


In [6]:
def ensure_online_raw_measurements_table(conn):
    conn.execute(
        """
        CREATE TABLE IF NOT EXISTS online_raw_measurements (
            city TEXT NOT NULL,
            sensor_id TEXT NOT NULL,
            timestamp_utc TEXT NOT NULL,
            timestamp_local TEXT,
            measurement_type TEXT NOT NULL,
            value REAL NOT NULL,
            lat REAL,
            lon REAL,
            source_file TEXT,
            UNIQUE (city, sensor_id, timestamp_utc, measurement_type)
        )
        """
    )
    conn.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_online_raw_measurements_timeline
        ON online_raw_measurements (city, measurement_type, timestamp_utc)
        """
    )
    conn.execute(
        """
        CREATE INDEX IF NOT EXISTS idx_online_raw_measurements_sensor_timeline
        ON online_raw_measurements (city, sensor_id, measurement_type, timestamp_utc)
        """
    )


def insert_online_raw_measurements(conn, measurements):
    records = list(
        measurements[[
            "city",
            "sensor_id",
            "timestamp_utc",
            "timestamp_local",
            "measurement_type",
            "value",
            "lat",
            "lon",
            "source_file",
        ]]
        .where(pd.notna(measurements), None)
        .itertuples(index=False, name=None)
    )

    conn.executemany(
        """
        INSERT OR REPLACE INTO online_raw_measurements (
            city,
            sensor_id,
            timestamp_utc,
            timestamp_local,
            measurement_type,
            value,
            lat,
            lon,
            source_file
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        records,
    )

    return len(records)

In [8]:
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(DB_PATH, timeout=30) as conn:
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA busy_timeout=30000")
    ensure_online_raw_measurements_table(conn)
    inserted = insert_online_raw_measurements(conn, raw_measurements)
    conn.commit()

print(f"Inserted/replaced {inserted} raw Pulse Eco rows into {DB_PATH}")

Inserted/replaced 154721 raw Pulse Eco rows into /mnt/c/Users/RazorVision/Desktop/project-vrnmp/data/bitola.db


In [10]:
with sqlite3.connect(DB_PATH) as conn:
    summary = pd.read_sql_query(
        """
        SELECT
            measurement_type,
            COUNT(*) AS rows,
            COUNT(DISTINCT sensor_id) AS sensors,
            MIN(timestamp_utc) AS min_timestamp_utc,
            MAX(timestamp_utc) AS max_timestamp_utc
        FROM online_raw_measurements
        WHERE city = ?
        GROUP BY measurement_type
        ORDER BY measurement_type
        """,
        conn,
        params=(CITY,),
    )

summary

,measurement_type,rows,sensors,min_timestamp_utc,max_timestamp_utc
0,pm10,77280,12,2025-12-01 00:00:43,2026-03-01 23:59:38
1,pm25,77441,12,2025-12-01 00:00:43,2026-03-01 23:59:38


## Example Streamlit query

Use this table as raw observed data. If a sensor/hour is missing, no row exists. Keep it separate from the imputed model feature table.

In [12]:
with sqlite3.connect(DB_PATH) as conn:
    example = pd.read_sql_query(
        """
        SELECT timestamp_utc, sensor_id, measurement_type, value
        FROM online_raw_measurements
        WHERE city = ?
          AND sensor_id = ?
          AND measurement_type IN ('pm10', 'pm25')
        ORDER BY timestamp_utc, measurement_type
        LIMIT 50
        """,
        conn,
        params=(CITY, "2002"),
    )

example

,timestamp_utc,sensor_id,measurement_type,value
0,2025-12-01 00:00:43,2002,pm10,115.0
1,2025-12-01 00:00:43,2002,pm25,103.0
2,2025-12-01 01:00:43,2002,pm10,78.0
3,2025-12-01 01:00:43,2002,pm25,70.0
4,2025-12-01 02:00:44,2002,pm10,62.0
5,2025-12-01 02:00:44,2002,pm25,58.0
6,2025-12-01 03:00:42,2002,pm10,51.0
7,2025-12-01 03:00:42,2002,pm25,48.0
8,2025-12-01 04:00:42,2002,pm10,48.0
9,2025-12-01 04:00:42,2002,pm25,45.0
